In [30]:
from dotenv import load_dotenv
from openai import OpenAI

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export import ConsoleSpanExporter

load_dotenv()
client = OpenAI()

In [31]:
from starter import rag as index, client

In [32]:
def setup_console_tracer():
    provider = TracerProvider()
    processor = SimpleSpanProcessor(ConsoleSpanExporter())
    provider.add_span_processor(processor)
    trace.set_tracer_provider(provider)
    return trace.get_tracer("llm-zoomcamp")

tracer = setup_console_tracer()

Overriding of current TracerProvider is not allowed


In [33]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = trace.get_tracer("llm-zoomcamp")
        
    def search(self, query, num_results=5):
        with self.tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            results = super().search(query, num_results)
            span.set_attribute("num_results_found", len(results))
            return results
    
    def llm(self, prompt):
        """Traced LLM call with metrics captured as attributes."""
        with self.tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            
            if hasattr(response, 'usage'):
                usage = response.usage
                input_tokens = usage.input_tokens
                output_tokens = usage.output_tokens
                total_tokens = usage.total_tokens
                
                cost = (input_tokens * 0.15 + output_tokens * 0.60) / 1_000_000
                
                span.set_attribute("input_tokens", input_tokens)
                span.set_attribute("output_tokens", output_tokens)
                span.set_attribute("total_tokens", total_tokens)
                span.set_attribute("cost", cost)
                span.set_attribute("model", self.model)
            
            return response
    
    def rag(self, query):
        with self.tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            
            results = self.search(query)
            context = "\n\n".join([r['content'] for r in results])
            prompt = f"Answer using this context:\n{context}\n\nQuestion: {query}"
            response = self.llm(prompt)
            
            if hasattr(response, 'output_text'):
                return response.output_text
            return response

rag_traced = RAGTraced(
    index=index,
    llm_client=client
)


In [34]:
QUERY = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(QUERY)

{
    "name": "search",
    "context": {
        "trace_id": "0x88e8cae370f958cf40ddc94b39350ae4",
        "span_id": "0x7bad42fc469ece9a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xfa594bb5b5db9641",
    "start_time": "2026-07-14T21:28:39.265882Z",
    "end_time": "2026-07-14T21:28:39.272386Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "num_results_found": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "c7d9de57-d2f0-43f0-92a3-6fae5221b158",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0x88e8cae370f958cf40ddc94b39350ae4",
        "span_id": "0x61b7514a4116398c",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xfa594bb5b5db9641",
    "start_time": "2026-07-14T21:28:39.276637Z",
    "end_time": "2026-07-14T21:28:40.756819Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7038,
        "output_tokens": 114,
        "total_tokens": 7152,
        "cost": 0.0011241,
        "model": "gpt-5.4-mini"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "c7d9de57-d2f0-43f0-92a3-6fae5221b158",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag",
    "context": {
    

In [35]:
answer = rag_traced.rag(QUERY)

{
    "name": "search",
    "context": {
        "trace_id": "0x1e5ee2456d3a6e3726a31c109cd249b8",
        "span_id": "0xf7e44b6007df6ab1",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x5dfa92a1eca91925",
    "start_time": "2026-07-14T21:28:40.777717Z",
    "end_time": "2026-07-14T21:28:40.786914Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "num_results_found": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "c7d9de57-d2f0-43f0-92a3-6fae5221b158",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        

In [36]:
answer = rag_traced.rag(QUERY)

{
    "name": "search",
    "context": {
        "trace_id": "0x4ba2460a7f51038116158cc06f7039a2",
        "span_id": "0xb27c1f386efec698",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x5e1189ef85c4fa02",
    "start_time": "2026-07-14T21:28:42.747010Z",
    "end_time": "2026-07-14T21:28:42.752133Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "num_results_found": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "c7d9de57-d2f0-43f0-92a3-6fae5221b158",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        

In [37]:
answer = rag_traced.rag(QUERY)

{
    "name": "search",
    "context": {
        "trace_id": "0xe58ce34f81ba97594267fab1ff4a4cba",
        "span_id": "0x5811d48a34017900",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb2a7d454c96845a3",
    "start_time": "2026-07-14T21:28:44.534441Z",
    "end_time": "2026-07-14T21:28:44.536958Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "num_results_found": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "c7d9de57-d2f0-43f0-92a3-6fae5221b158",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        